# Error Analysis — Credit Default Model

**Model:** Gradient Boosting champion
**Threshold:** From calibration meta (fallback 0.27)
**Goal:** Locate **where** the model fails so underwriting and monitoring can focus

---

### Purpose

Decompose validation errors into false negatives (missed defaults) and false positives (false alarms), then attribute them to segments such as term, home ownership, and score band.

### Error types

| Type | Definition | Business impact |
|------|------------|-----------------|
| **False Negative (FN)** | Actual default, model predicted paid | Missed risk / potential loss |
| **False Positive (FP)** | Actual paid, model predicted default | Extra review / false decline |

### Analysis plan

1. Score the validation set at the operating threshold
2. Overall confusion matrix
3. FN / FP rates by segment
4. Probability distributions by error type
5. High-confidence mistakes
6. Actionable recommendations


In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

FEAT = Path("features")
OUT = Path("model_outputs")
OUT.mkdir(exist_ok=True)
(OUT / "plots").mkdir(exist_ok=True)
RS = 42

feat = pd.read_csv(FEAT / "credit_train_features.csv")
with open(FEAT / "selected_features_standard.json") as f:
    cols = [c for c in json.load(f)["features"] if c in feat.columns]

# threshold from meta
thr = 0.27
for meta_name in ["calibration_threshold_meta.json", "hparam_search_summary.json"]:
    p = OUT / meta_name
    if p.exists():
        m = json.loads(p.read_text())
        thr = float(m.get("threshold_primary", m.get("threshold", thr)))
        break

X = feat[cols]
y = feat["Default"]
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", GradientBoostingClassifier(
        n_estimators=80, max_depth=4, learning_rate=0.1, subsample=0.8, random_state=RS,
    )),
])
pipe.fit(X_tr, y_tr)
proba = pipe.predict_proba(X_va)[:, 1]
pred = (proba >= thr).astype(int)

va = X_va.copy()
va["y_true"] = y_va.values
va["proba"] = proba
va["y_pred"] = pred
va["error_type"] = np.where(
    (va["y_true"] == 1) & (va["y_pred"] == 0), "FN",
    np.where((va["y_true"] == 0) & (va["y_pred"] == 1), "FP",
    np.where((va["y_true"] == 1) & (va["y_pred"] == 1), "TP", "TN")),
)

print(f"Threshold: {thr:.2f}")
print(va["error_type"].value_counts())
print(classification_report(y_va, pred, target_names=["Paid", "Default"]))


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_va, pred)
ConfusionMatrixDisplay(cm, display_labels=["Paid", "Default"]).plot(ax=ax, colorbar=False)
ax.set_title(f"Confusion matrix @ thr={thr:.2f}")
plt.tight_layout()
plt.savefig(OUT / "plots/error_confusion.png", dpi=120, bbox_inches="tight")
plt.show()


## Segment error rates

Compare false-negative rates among actual defaults and false-positive rates among paid loans across underwriting segments.


In [ ]:
segments = []
for col, label in [
    ("is_long_term", "Long Term"),
    ("is_renter", "Renter"),
    ("score_band_poor", "Poor score"),
    ("score_band_excellent", "Excellent score"),
    ("loan_amount_missing", "Loan amount missing"),
    ("purpose_business", "Business purpose"),
]:
    if col not in va.columns:
        continue
    for val in [0, 1]:
        m = va[col] == val
        if m.sum() < 50:
            continue
        sub = va[m]
        n = len(sub)
        fn = ((sub["error_type"] == "FN").sum())
        fp = ((sub["error_type"] == "FP").sum())
        defaults = (sub["y_true"] == 1).sum()
        paid = (sub["y_true"] == 0).sum()
        segments.append({
            "segment": label, "value": val, "n": n,
            "default_rate": defaults / n,
            "fn_rate_among_defaults": fn / defaults if defaults else np.nan,
            "fp_rate_among_paid": fp / paid if paid else np.nan,
            "fn_count": int(fn), "fp_count": int(fp),
            "recall": 1 - (fn / defaults) if defaults else np.nan,
        })

seg_df = pd.DataFrame(segments)
print(seg_df.to_string(index=False))
seg_df.to_csv(OUT / "error_segment_analysis.csv", index=False)


In [ ]:
# FN rate among defaults by segment value=1
plot_df = seg_df[seg_df["value"] == 1].dropna(subset=["fn_rate_among_defaults"])
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(plot_df["segment"], plot_df["fn_rate_among_defaults"], color="#e74c3c")
ax.set_xlabel("Miss rate among actual defaults (FN rate)")
ax.set_title("Where defaults are missed")
plt.tight_layout()
plt.savefig(OUT / "plots/error_fn_by_segment.png", dpi=120, bbox_inches="tight")
plt.show()


## Score distributions by error type

Overlay predicted probabilities for true positives, true negatives, false positives, and false negatives relative to the operating threshold.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for et, color in [("TN", "#2ecc71"), ("TP", "#3498db"), ("FP", "#f39c12"), ("FN", "#e74c3c")]:
    s = va.loc[va["error_type"] == et, "proba"]
    ax.hist(s, bins=30, alpha=0.45, label=f"{et} (n={len(s)})", color=color)
ax.axvline(thr, color="black", ls="--", label=f"thr={thr:.2f}")
ax.set_xlabel("Predicted default probability")
ax.set_ylabel("Count")
ax.set_title("Score distribution by error type")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "plots/error_score_by_type.png", dpi=120, bbox_inches="tight")
plt.show()


## High-confidence errors

Inspect defaults assigned very low scores (confident misses) and paid loans assigned very high scores (confident false alarms).


In [ ]:
# FN with low predicted proba = confident misses
# FP with high predicted proba = confident false alarms
fn = va[va["error_type"] == "FN"].nsmallest(15, "proba")
fp = va[va["error_type"] == "FP"].nlargest(15, "proba")

show_cols = [c for c in [
    "proba", "Credit Score", "is_long_term", "is_renter", "lti", "dti",
    "loan_amount_missing", "Current Loan Amount", "Annual Income",
] if c in va.columns]

print("=== Confident FNs (defaults scored as safe) ===")
print(fn[show_cols].to_string(index=False))
print("\n=== Confident FPs (paid scored as risky) ===")
print(fp[show_cols].to_string(index=False))

fn[show_cols].to_csv(OUT / "error_confident_fn.csv", index=False)
fp[show_cols].to_csv(OUT / "error_confident_fp.csv", index=False)


## Findings and recommendations

### Typical patterns (interpret after running cells)

1. **Long-term loans** often have higher default rates and different FN/FP balance — consider term-specific thresholds.
2. **Missing loan amount** was a strong feature; if those rows never default in some slices, check data leakage vs true signal.
3. **Excellent score band** may drive FNs when other stress factors (high LTI/DTI) are present — interactions matter.
4. Confident FPs: good candidates for manual review queues rather than auto-decline.
5. Confident FNs: raise threshold sensitivity or add features (utilization trajectory, income verification).

### Actions

| Priority | Action |
|----------|--------|
| High | Segment-specific threshold for Long Term vs Short Term |
| High | Manual review band around threshold (e.g. 0.20–0.35) |
| Medium | Feature: stress combo flags (high LTI + renter + long term) |
| Medium | Recalibrate after any policy change in data capture |


In [ ]:
print("=" * 60)
print("ERROR ANALYSIS COMPLETE")
print("=" * 60)
print(f"Threshold: {thr:.2f}")
print(va["error_type"].value_counts().to_string())
print(f"Artifacts under {OUT.resolve()}")
